# Week 36, Day 2: Command and undo

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w036d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
done = []
text = []

def type_word(word):
    text.append(word)
    done.append(word)

def undo():
    done.pop()
    text.pop()

for w in ["Ẹ", "káàbọ̀", "everyone"]:
    type_word(w)
undo()
type_word("all")
print(" ".join(text), len(done))

In [ ]:
class Account:
    def __init__(self, balance=0):
        self.balance = balance

class Deposit:
    def __init__(self, account, amount):
        self.account = account
        self.amount = amount

    def execute(self):
        self.account.balance += self.amount

    def undo(self):
        self.account.balance -= self.amount

    def __repr__(self):
        return f"Deposit({self.amount})"

acc = Account(100)
d = Deposit(acc, 50)
d.execute()
print(acc.balance, d)
d.undo()
print(acc.balance)

In [ ]:
class History:
    def __init__(self):
        self.undo_stack = []
        self.redo_stack = []

    def run(self, command):
        command.execute()
        self.undo_stack.append(command)
        self.redo_stack.clear()

    def undo(self):
        if not self.undo_stack:
            return False
        command = self.undo_stack.pop()
        command.undo()
        self.redo_stack.append(command)
        return True

    def redo(self):
        if not self.redo_stack:
            return False
        command = self.redo_stack.pop()
        command.execute()
        self.undo_stack.append(command)
        return True

class Account:
    def __init__(self, balance=0):
        self.balance = balance

class Deposit:
    def __init__(self, account, amount):
        self.account, self.amount = account, amount
    def execute(self):
        self.account.balance += self.amount
    def undo(self):
        self.account.balance -= self.amount

acc = Account()
h = History()
for amount in [100, 50, 25]:
    h.run(Deposit(acc, amount))
h.undo()
h.undo()
print(acc.balance)
h.redo()
print(acc.balance)
h.run(Deposit(acc, 1))           # a new action: the redo stack is cleared
print(acc.balance, h.redo())

In [ ]:
class Document:
    def __init__(self, text=""):
        self.text = text

class Replace:
    def __init__(self, doc, old, new):
        self.doc, self.old, self.new = doc, old, new
        self.before = None                    # filled in by execute

    def execute(self):
        self.before = self.doc.text
        self.doc.text = self.doc.text.replace(self.old, self.new)

    def undo(self):
        self.doc.text = self.before

class Delete:
    def __init__(self, doc, start, end):
        self.doc, self.start, self.end = doc, start, end
        self.removed = ""

    def execute(self):
        self.removed = self.doc.text[self.start:self.end]
        self.doc.text = self.doc.text[:self.start] + self.doc.text[self.end:]

    def undo(self):
        self.doc.text = self.doc.text[:self.start] + self.removed + self.doc.text[self.start:]

doc = Document("fish & chips & peas")
r = Replace(doc, "&", "and")
r.execute()
d = Delete(doc, 0, 5)
d.execute()
print(repr(doc.text))
d.undo()
r.undo()
print(repr(doc.text))

In [ ]:
class Macro:
    def __init__(self, commands):
        self.commands = list(commands)

    def execute(self):
        for c in self.commands:
            c.execute()

    def undo(self):
        for c in reversed(self.commands):
            c.undo()

class Account:
    def __init__(self, balance=0):
        self.balance = balance

class Deposit:
    def __init__(self, account, amount):
        self.account, self.amount = account, amount
    def execute(self):
        self.account.balance += self.amount
    def undo(self):
        self.account.balance -= self.amount

savings, current = Account(500), Account(100)
transfer = Macro([Deposit(savings, -200), Deposit(current, 200)])
transfer.execute()
print(savings.balance, current.balance)
transfer.undo()
print(savings.balance, current.balance)

In [ ]:
def append_item(items, item):
    items.append(item)
    return lambda: items.pop()

def set_key(d, key, value):
    had, old = key in d, d.get(key)
    d[key] = value
    def undo():
        if had:
            d[key] = old
        else:
            del d[key]
    return undo

basket, prices = [], {"rice": 450}
undos = [append_item(basket, "rice"), set_key(prices, "rice", 500), set_key(prices, "oil", 900)]
print(basket, prices)
while undos:
    undos.pop()()
print(basket, prices)

> **Key idea.** Command (Gang of Four): turn an action into an object with `execute()` and `undo()`, holding its receiver, settings and whatever it must remember to reverse itself. Actions as objects can be stored, undone, redone, queued, logged and replayed. Undo and redo use two stacks: running a command pushes it on the undo stack and clears the redo stack; undo moves the top command to the redo stack, redo moves it back. Commands that destroy information save it when they execute (just the change, or a snapshot). A macro holds commands and undoes them in reverse. A function that does an action and returns its undo is a lighter form without redo.

### Exercise: Commands for a counter

Write two command classes for a `Counter` (given): `Add(counter, n)` and `Reset(counter)`. Both have `execute()` and `undo()`. `Reset` sets the value to 0, and its `undo` must restore whatever the value was before it ran.

In [ ]:
class Counter:
    def __init__(self, value=0):
        self.value = value

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_commands():\n    \"do and undo\"\n    C, A, R = ns[\"Counter\"], ns[\"Add\"], ns[\"Reset\"]\n    c = C(5)\n    a, r = A(c, 3), R(c)\n    a.execute()\n    r.execute()\n    assert c.value == 0\n    r.undo()\n    assert c.value == 8\n    a.undo()\n    assert c.value == 5\n    r2 = R(c)\n    c.value = 42\n    r2.execute()\n    r2.undo()\n    assert c.value == 42, \"Reset should remember the value when it runs, not when it's made\"")

### Exercise: A history with undo and redo

Write `History` with `run(command)`, `undo()` and `redo()` (each returning `True` if it did something, `False` if there was nothing to undo or redo), and properties `can_undo` and `can_redo`. Running a new command clears the redo stack. It also has `labels()`, a list of `str(command)` for the commands that can currently be undone, oldest first.

In [ ]:
class Counter:
    def __init__(self):
        self.value = 0

class Add:
    def __init__(self, counter, n):
        self.counter, self.n = counter, n
    def execute(self):
        self.counter.value += self.n
    def undo(self):
        self.counter.value -= self.n
    def __str__(self):
        return f"add {self.n}"

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_history():\n    \"undo and redo\"\n    C, A, H = ns[\"Counter\"], ns[\"Add\"], ns[\"History\"]\n    c, h = C(), H()\n    assert not h.can_undo and not h.undo() and not h.redo()\n    for n in [1, 10, 100]:\n        h.run(A(c, n))\n    assert c.value == 111 and h.labels() == [\"add 1\", \"add 10\", \"add 100\"]\n    assert h.undo() and h.undo() and c.value == 1 and h.can_redo and h.labels() == [\"add 1\"]\n    assert h.redo() and c.value == 11\n    h.run(A(c, 5))\n    assert c.value == 16 and not h.can_redo and not h.redo() and h.labels() == [\"add 1\", \"add 10\", \"add 5\"]\n    while h.undo():\n        pass\n    assert c.value == 0 and not h.can_undo")

### Exercise: Editing commands

A `Buffer` holds `text`. Write three commands: `Insert(buffer, pos, s)`, `Delete(buffer, start, end)` (which remembers what it removed) and `Upper(buffer, start, end)` (capitalises that slice, remembering what was there). Each has `execute` and `undo`. A position outside the text (below 0 or beyond its length) raises `ValueError` when the command **executes**, and changes nothing. After any sequence of commands, undoing them in reverse order must give back the original text.

In [ ]:
class Buffer:
    def __init__(self, text=""):
        self.text = text

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_editing():\n    \"edits and undo\"\n    B, I, D, U = ns[\"Buffer\"], ns[\"Insert\"], ns[\"Delete\"], ns[\"Upper\"]\n    b = B(\"rail strike called off\")\n    cmds = [I(b, 0, \"BREAKING: \"), U(b, 10, 14), D(b, 15, 22)]\n    for c in cmds:\n        c.execute()\n    assert b.text == \"BREAKING: RAIL called off\", f\"got {b.text!r}\"\n    for c in reversed(cmds):\n        c.undo()\n    assert b.text == \"rail strike called off\"\n    for bad in [I(b, 99, \"x\"), D(b, -1, 3), U(b, 0, 100)]:\n        try:\n            bad.execute()\n            assert False, \"an out-of-range position should raise ValueError\"\n        except ValueError:\n            pass\n    assert b.text == \"rail strike called off\"")

### Exercise: All or nothing

Write `Macro(commands)`, a command whose `execute` runs its commands in order and whose `undo` undoes them in reverse. If one of them raises an exception during `execute`, the macro must **undo the ones that already ran** (in reverse) and then let the error through, so a failed macro leaves everything as it was. Then use it in `transfer(source, target, amount)`, which returns a macro of two `Change` commands (given) that moves money between two accounts; a `Change` that would make a balance negative raises `ValueError`.

In [ ]:
class Account:
    def __init__(self, balance):
        self.balance = balance

class Change:
    def __init__(self, account, amount):
        self.account, self.amount = account, amount
    def execute(self):
        if self.account.balance + self.amount < 0:
            raise ValueError("not enough money")
        self.account.balance += self.amount
    def undo(self):
        self.account.balance -= self.amount

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_macro():\n    \"all or nothing\"\n    A, M = ns[\"Account\"], ns[\"Macro\"]\n    a, b = A(500), A(100)\n    t = transfer(a, b, 200)\n    t.execute()\n    assert (a.balance, b.balance) == (300, 300)\n    t.undo()\n    assert (a.balance, b.balance) == (500, 100)\n    log = []\n    class Step:\n        def __init__(self, name, fail=False):\n            self.name, self.fail = name, fail\n        def execute(self):\n            if self.fail:\n                raise RuntimeError(\"boom\")\n            log.append(\"do \" + self.name)\n        def undo(self):\n            log.append(\"undo \" + self.name)\n    try:\n        M([Step(\"a\"), Step(\"b\"), Step(\"c\", fail=True), Step(\"d\")]).execute()\n        assert False, \"the error should come through\"\n    except RuntimeError:\n        pass\n    assert log == [\"do a\", \"do b\", \"undo b\", \"undo a\"], f\"log: {log}\"\n    try:\n        transfer(b, a, 1000).execute()\n        assert False\n    except ValueError:\n        pass\n    assert (a.balance, b.balance) == (500, 100)")

### Bug hunt: A redo from a different past

After undoing, the user ran a new command, then pressed redo, and an old command came back on top of the new one: this prints `103`. A new command must clear the redo stack. Fix `run` so redo does nothing and the program prints `3`.

In [ ]:
class Counter:
    def __init__(self):
        self.value = 0

class Add:
    def __init__(self, c, n):
        self.c, self.n = c, n
    def execute(self):
        self.c.value += self.n
    def undo(self):
        self.c.value -= self.n

class History:
    def __init__(self):
        self.undos, self.redos = [], []
    def run(self, cmd):
        cmd.execute()
        self.undos.append(cmd)
    def undo(self):
        if self.undos:
            cmd = self.undos.pop()
            cmd.undo()
            self.redos.append(cmd)
    def redo(self):
        if self.redos:
            cmd = self.redos.pop()
            cmd.execute()
            self.undos.append(cmd)

c, h = Counter(), History()
h.run(Add(c, 100))
h.undo()
h.run(Add(c, 3))
h.redo()
print(c.value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a fresh future\"\n    expect_output(\"3\")")

### Bug hunt: Remembered too early

`SetPrice` saves the old price when it's **made**, but the price changes before it runs, so undo restores the wrong value: this prints `450`. Fix the command so it remembers the price when it executes, and the program prints `500`.

In [ ]:
class SetPrice:
    def __init__(self, prices, item, new):
        self.prices, self.item, self.new = prices, item, new
        self.old = prices[item]
    def execute(self):
        self.prices[self.item] = self.new
    def undo(self):
        self.prices[self.item] = self.old

prices = {"rice": 450}
cmd = SetPrice(prices, "rice", 600)
prices["rice"] = 500
cmd.execute()
cmd.undo()
print(prices["rice"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the right old value\"\n    expect_output(\"500\")")

### Bug hunt: Undone in the wrong order

The macro undoes its steps in the same order it did them, so the second insertion is removed from the wrong place (its position only makes sense while the first insertion is still there): this prints `'big wor'`. Fix `Macro.undo` so the program prints `'world'`.

In [ ]:
class Doc:
    def __init__(self, text):
        self.text = text

class Insert:
    def __init__(self, d, pos, s):
        self.d, self.pos, self.s = d, pos, s
    def execute(self):
        self.d.text = self.d.text[:self.pos] + self.s + self.d.text[self.pos:]
    def undo(self):
        self.d.text = self.d.text[:self.pos] + self.d.text[self.pos + len(self.s):]

class Macro:
    def __init__(self, cmds):
        self.cmds = cmds
    def execute(self):
        for c in self.cmds:
            c.execute()
    def undo(self):
        for c in self.cmds:
            c.undo()

d = Doc("world")
m = Macro([Insert(d, 0, "Hello, "), Insert(d, 7, "big ")])
m.execute()
m.undo()
print(repr(d.text))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"reverse order\"\n    expect_output(\"'world'\")")

### Puzzle: Undo with two stacks

Arrange the lines into `undo` for a history with an undo stack and a redo stack. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
log = []

def put(lst, x):
    lst.append(x)
    return lambda: lst.remove(x)

items = []
undos = [put(items, "a"), put(items, "b"), put(items, "c")]
undos[1]()
log.append(list(items))
undos.pop()()
log.append(list(items))

p = ___   # log[0]
q = ___   # log[1]
r = ___   # len(undos)
s = ___   # items

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q) == ([\"a\", \"c\"], [\"a\"]), \"p and q\"\n    assert (r, s) == (2, [\"a\"])")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a drawing with undo

A tiny drawing program keeps shapes in a `Canvas` (a dictionary `shapes` from a shape's id to a dictionary of its properties, such as `{"kind": "circle", "x": 10, "y": 20, "colour": "red"}`). Write commands and an `Editor`:

- `AddShape(canvas, shape_id, props)`: adds a shape (a copy of `props`); an id already in use raises `ValueError`.
- `Move(canvas, shape_id, dx, dy)`: moves a shape; an unknown id raises `KeyError`.
- `Recolour(canvas, shape_id, colour)`: changes its colour, remembering the old one.
- `RemoveShape(canvas, shape_id)`: removes it, remembering it (and its position in the canvas's order, so undo puts it back in the same place).
- `Editor(canvas)` with `do(command)`, `undo()`, `redo()` (as in today's `History`), and `group(commands)`, which runs several commands as **one** undoable step (all or nothing: if one fails, the ones before it are undone and the error comes through).

Every command's undo must restore the canvas exactly, including the order of `shapes`.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_drawing():\n    \"drawing\"\n    C, E, Add, Mv, Rc, Rm = (ns[n] for n in [\"Canvas\", \"Editor\", \"AddShape\", \"Move\", \"Recolour\", \"RemoveShape\"])\n    c = C()\n    e = E(c)\n    props = {\"kind\": \"circle\", \"x\": 0, \"y\": 0, \"colour\": \"red\"}\n    e.do(Add(c, \"a\", props))\n    props[\"x\"] = 999\n    e.do(Add(c, \"b\", {\"kind\": \"square\", \"x\": 5, \"y\": 5, \"colour\": \"blue\"}))\n    e.do(Add(c, \"c\", {\"kind\": \"line\", \"x\": 1, \"y\": 1, \"colour\": \"black\"}))\n    e.do(Mv(c, \"a\", 3, -2))\n    e.do(Rc(c, \"b\", \"green\"))\n    assert c.shapes[\"a\"][\"x\"] == 3 and c.shapes[\"a\"][\"y\"] == -2 and c.shapes[\"b\"][\"colour\"] == \"green\"\n    e.do(Rm(c, \"b\"))\n    assert list(c.shapes) == [\"a\", \"c\"]\n    e.undo()\n    assert list(c.shapes) == [\"a\", \"b\", \"c\"] and c.shapes[\"b\"][\"colour\"] == \"green\"\n    e.undo()\n    e.undo()\n    assert c.shapes[\"b\"][\"colour\"] == \"blue\" and c.shapes[\"a\"][\"x\"] == 0\n    e.redo()\n    assert c.shapes[\"a\"][\"x\"] == 3\ndef test_group():\n    \"grouped steps\"\n    C, E, Add, Mv, Rc = (ns[n] for n in [\"Canvas\", \"Editor\", \"AddShape\", \"Move\", \"Recolour\"])\n    c = C()\n    e = E(c)\n    e.group([Add(c, \"a\", {\"kind\": \"dot\", \"x\": 0, \"y\": 0, \"colour\": \"red\"}), Mv(c, \"a\", 1, 1), Rc(c, \"a\", \"gold\")])\n    assert c.shapes[\"a\"] == {\"kind\": \"dot\", \"x\": 1, \"y\": 1, \"colour\": \"gold\"}\n    e.undo()\n    assert c.shapes == {}\n    e.redo()\n    assert c.shapes[\"a\"][\"colour\"] == \"gold\"\n    for bad, err in [([Mv(c, \"a\", 5, 5), Mv(c, \"nope\", 1, 1)], KeyError), ([Add(c, \"a\", {\"kind\": \"x\", \"x\": 0, \"y\": 0, \"colour\": \"r\"})], ValueError)]:\n        try:\n            e.group(bad)\n            assert False, \"should raise\"\n        except err:\n            pass\n    assert c.shapes[\"a\"][\"x\"] == 1, \"a failed group leaves everything as it was\"\n    e.undo()\n    assert c.shapes == {}")

## Warm-up

To undo the last word, the program had to **remember** what was done, in order. Here that was easy, because every action was "add a word". Real programs have many kinds of action (type, delete, bold, move), each undone differently. The Command pattern makes each action an object that knows how to undo itself.

## Actions as objects

The Gang of Four describe Command as: "encapsulate a request as an object, thereby letting you parameterise clients with different requests, queue or log requests, and support undoable operations." A command object has:

- an `execute()` method, which carries out the action;
- an `undo()` method, which reverses it;
- whatever it needs to do both: what it acts on (the **receiver**), its settings, and anything it must remember to reverse itself.

Once actions are objects, a program can do things with them that it can't do with plain function calls: keep a **history**, **undo** and **redo** them, **queue** them to run later, **log** them, send them to another program, or **replay** them (to rebuild a document from its history, say).

## Undo and redo

Undo and redo need two stacks (lists used last-in, first-out, Week 8):

- **run** a command: execute it, push it onto the undo stack, and **clear the redo stack** (a new action makes the old "future" meaningless);
- **undo**: pop from the undo stack, undo it, push it onto the redo stack;
- **redo**: pop from the redo stack, execute it again, push it back onto the undo stack.

The `History` class knows nothing about accounts: it works with any objects that have `execute` and `undo` (duck typing again). The same history can manage commands for text, drawings or bank accounts.

## What a command must remember

A deposit is easy to undo: subtract what was added. Other actions **destroy information**, and the command must save it when it runs, so it can put it back:

Two ways to remember:

- **Save just what changed** (`Delete` keeps the removed characters). Small and precise, but you must get the reverse exactly right.
- **Save a snapshot** of the state before (`Replace` keeps the whole text). Simple and hard to get wrong, but costly for large data. (Keeping snapshots is sometimes called the Memento pattern.)

Either way, commands must be undone in **reverse order**: `Delete`'s positions only make sense on the text as it was after `Replace` ran. The undo stack guarantees that.

## Macros

A **macro command** groups several commands so they're done and undone as one: "format as heading" might be bold plus bigger plus centred, and one undo should reverse all three. It's a command that holds a list of commands:

A macro undoes its parts in **reverse** order, for the same reason a history does. Because a macro has `execute` and `undo`, it's a command like any other, so it can go in a history, or even inside another macro.

## Commands as functions

When commands don't need names, reprs or settings that others inspect, a lighter Python form works: a function that **does** the action and **returns a function that undoes it**:

The closure (Week 7) remembers what's needed to undo. This form is compact, but it can't redo (the "do" part isn't kept) and can't describe itself in a log. Use classes when you need redo, history display or saving commands; use functions for quick, local undo.

## Practice

## Quiz

## Daily challenge

## Before you go

Think about a program you use every day and list three actions it lets you undo, and one it doesn't. Why might that one be hard to undo? Tomorrow: the Iterator pattern, the protocol behind every `for` loop.